<a href="https://colab.research.google.com/github/Yukselendincer/datasceinceproject/blob/main/ETL_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
import pandas as pd

def extract_crypto_data():
    # CoinGecko Public API (API Key gerekmez, ancak saniyede 1-2 istek sınırı vardır)
    url = "https://api.coingecko.com/api/v3/coins/markets"

    params = {
        "vs_currency": "usd",      # Dolar cinsinden fiyat
        "order": "market_cap_desc", # Piyasa değerine göre sırala
        "per_page": 10,            # İlk 10 coin
        "page": 1,
        "sparkline": "false"       # Grafik verisi istemiyoruz (şimdilik)
    }

    try:
        response = requests.get(url, params=params)
        response.raise_for_status() # Hata kontrolü
        data = response.json()
        print(f"Başarıyla {len(data)} adet coin verisi çekildi.")
        return data
    except Exception as e:
        print(f"Hata oluştu: {e}")
        return None

# Test edelim
raw_data = extract_crypto_data()

In [ ]:
df_crypto = pd.DataFrame(raw_data)
df_crypto.head()

In [ ]:
df_crypto.info()

Here's a breakdown of the columns in the `df_crypto` DataFrame:

*   **id**: Unique identifier for the cryptocurrency (e.g., 'bitcoin', 'ethereum').
*   **symbol**: The ticker symbol of the cryptocurrency (e.g., 'btc', 'eth').
*   **name**: The full name of the cryptocurrency (e.g., 'Bitcoin', 'Ethereum').
*   **image**: URL of the cryptocurrency's logo.
*   **current_price**: The current price of the cryptocurrency in USD.
*   **market_cap**: The total market capitalization of the cryptocurrency.
*   **market_cap_rank**: The ranking of the cryptocurrency by market capitalization.
*   **fully_diluted_valuation**: The fully diluted market capitalization, assuming all future coins are in circulation.
*   **total_volume**: The total trading volume in the last 24 hours.
*   **high_24h**: The highest price reached in the last 24 hours.
*   **low_24h**: The lowest price reached in the last 24 hours.
*   **price_change_24h**: The change in price over the last 24 hours.
*   **price_change_percentage_24h**: The percentage change in price over the last 24 hours.
*   **market_cap_change_24h**: The change in market capitalization over the last 24 hours.
*   **market_cap_change_percentage_24h**: The percentage change in market capitalization over the last 24 hours.
*   **circulating_supply**: The number of coins that are publicly available and circulating in the market.
*   **total_supply**: The total number of coins that exist now or will ever exist.
*   **max_supply**: The theoretical maximum number of coins that can ever exist.
*   **ath**: All-Time High price.
*   **ath_change_percentage**: Percentage difference from the All-Time High price.
*   **ath_date**: Date when the All-Time High price was reached.
*   **atl**: All-Time Low price.
*   **atl_change_percentage**: Percentage difference from the All-Time Low price.
*   **atl_date**: Date when the All-Time Low price was reached.
*   **roi**: Return on Investment information (can be complex, often a dictionary).
*   **last_updated**: The timestamp when the data was last updated.

🔑 Temel Kimlik Bilgileri
id: Kripto paranın sistemdeki benzersiz adı (Örn: bitcoin).

symbol: Kısa borsa kodu (Örn: btc, eth).

name: Kripto paranın tam adı (Örn: Bitcoin).

image: Logonun bulunduğu internet adresi (URL). Not: Genelde ETL projelerinde bu sütun temizlik aşamasında silinir.

💰 Fiyat ve Piyasa Verileri
current_price: Güncel fiyat (USD cinsinden).

market_cap: Toplam piyasa değeri (Dolaşımdaki arz × Güncel fiyat).

market_cap_rank: Piyasa değerine göre dünya sıralaması.

fully_diluted_valuation (FDV): Tam seyreltilmiş değer. Eğer tüm coinler piyasada olsaydı toplam değer ne olurdu sorusunun cevabı.

total_volume: Son 24 saatteki toplam işlem hacmi.

📉 Değişim ve İstatistikler (Transformation İçin Önemli)
high_24h / low_24h: Son 24 saat içindeki en yüksek ve en düşük fiyat.

price_change_24h: Son 24 saatteki fiyat değişimi (Miktar olarak).

price_change_percentage_24h: Son 24 saatteki fiyat değişimi (Yüzde olarak).

market_cap_change_24h: Piyasa değerindeki son 24 saatlik değişim.

🪙 Arz Bilgileri
circulating_supply: Şu an piyasada dolaşan coin miktarı.

total_supply: Üretilmiş olan toplam coin miktarı.

max_supply: Teorik olarak gelecekte var olabilecek maksimum miktar. (Bazı coinlerde null olabilir, bu ETL'de temizlenmelidir).

🏆 Tarihi Rekorlar (ATH & ATL)
ath (All-Time High): Tüm zamanların en yüksek fiyatı.

ath_change_percentage: Mevcut fiyatın zirveden ne kadar uzakta olduğu (Yüzde).

ath_date: Rekorun kırıldığı tarih.

atl (All-Time Low): Tüm zamanların en düşük fiyatı.

⚙️ Teknik Detaylar
roi (Return on Investment): Yatırım getirisi. (Genelde bir sözlük/dictionary formatında gelir, ETL'de bunu düzleştirmek gerekebilir).

last_updated: Verinin en son ne zaman güncellendiği (Zaman damgası).

In [ ]:
import pandas as pd
from datetime import datetime

def transform_crypto_data(raw_data):
    if not raw_data:
        return None

    # 1. Veriyi DataFrame'e çevir
    df = pd.DataFrame(raw_data)

    # 2. Sadece ihtiyacımız olan sütunları seçelim (Gürültüyü azaltıyoruz)
    columns_to_keep = [
        'id', 'symbol', 'name', 'current_price', 'market_cap',
        'total_volume', 'high_24h', 'low_24h', 'price_change_percentage_24h',
        'circulating_supply', 'ath', 'ath_date', 'last_updated'
    ]
    df = df[columns_to_keep]

    # 3. Veri Tiplerini Düzenle (Cleaning)
    # Tarihleri datetime objesine çevirelim
    df['ath_date'] = pd.to_datetime(df['ath_date'])
    df['last_updated'] = pd.to_datetime(df['last_updated'])

    # 4. Yeni Metrikler Türetelim (Feature Engineering)
    # Gün içi volatilite (En yüksek ve en düşük fiyat farkı yüzdesi)
    df['daily_volatility'] = ((df['high_24h'] - df['low_24h']) / df['low_24h']) * 100

    # Fiyatları daha okunabilir hale getirmek için yuvarlama
    df['current_price'] = df['current_price'].round(4)
    df['daily_volatility'] = df['daily_volatility'].round(2)

    # 5. Eksik Veri Kontrolü (Handling Missing Values)
    # Bazı yeni coinlerde ath_date boş gelebilir, onları "Bilinmiyor" veya uygun bir değerle doldurabiliriz
    df['ath_date'] = df['ath_date'].fillna(datetime.now())

    print("Dönüştürme işlemi başarıyla tamamlandı.")
    return df

# Akışı çalıştıralım (Önceki adımdaki raw_data'yı kullanıyoruz)
df_cleaned = transform_crypto_data(raw_data)
print(df_cleaned.head())

In [ ]:
import sqlite3

def load_crypto_data(df, db_name="crypto_warehouse.db"):
    if df is None:
        print("Yüklenecek veri bulunamadı!")
        return

    # 1. Veritabanına bağlan (Dosya yoksa otomatik oluşturulur)
    conn = sqlite3.connect(db_name)

    try:
        # 2. DataFrame'i SQL tablosuna yaz
        # if_exists='replace': Tablo varsa üstüne yazar (Geliştirme aşaması için ideal)
        # index=False: Pandas index'ini sütun olarak eklemez
        df.to_sql('top_cryptos', conn, if_exists='replace', index=False)

        print(f"Başarılı! Veriler '{db_name}' içindeki 'top_cryptos' tablosuna yüklendi.")

        # 3. Küçük bir test sorgusu yapalım
        query = "SELECT name, current_price, daily_volatility FROM top_cryptos LIMIT 5"
        test_df = pd.read_sql(query, conn)
        print("\nVeritabanından ilk 5 kayıt (Test):")
        print(test_df)

    except Exception as e:
        print(f"Yükleme sırasında hata: {e}")
    finally:
        # Bağlantıyı kapat
        conn.close()

# Süreci tamamla
load_crypto_data(df_cleaned)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Grafik stilini modern bir görünüme sokalım
sns.set_theme(style="whitegrid")

In [ ]:
def visualize_crypto_market_cap(df):
    plt.figure(figsize=(12, 6))

    # Çubuk grafik (Barplot)
    plot = sns.barplot(
        data=df,
        x='market_cap',
        y='name',
        palette='viridis'
    )

    plt.title('Top 10 Kripto Para: Piyasa Değeri Sıralaması', fontsize=15, fontweight='bold')
    plt.xlabel('Piyasa Değeri (USD)', fontsize=12)
    plt.ylabel('Kripto Para', fontsize=12)

    # Eksen değerlerini daha okunabilir yapalım (Milyar dolar formatı gibi)
    plt.ticklabel_format(style='plain', axis='x')

    plt.tight_layout()
    plt.show()

visualize_crypto_market_cap(df_cleaned)

In [ ]:
# Gerekli kütüphaneleri kur
!pip install mlflow yfinance dagshub --quiet

import mlflow
import yfinance as yf

# MLflow'u uzak bir sunucuya (örneğin Dagshub) bağlamak profesyonel bir MLOps adımıdır
# Buraya kendi linkini ekleyeceksin
# mlflow.set_tracking_uri("https://dagshub.com/kullanici_adin/proje_adin.mlflow")

with mlflow.start_run(run_name="Volatilite_Model_V1"):
    # 1. Veri Çekme ve Integrity Check
    data = yf.download("BTC-USD", period="1y")
    mlflow.log_param("ticker", "BTC-USD")

    # 2. Basit bir Volatilite Hesaplama (Örn: 7 günlük standart sapma)
    data['volatility'] = data['Close'].pct_change().rolling(7).std()

    # 3. Model Eğitimi (Burada basit bir regresyon olduğunu varsayalım)
    # score = model.fit(...)

    # 4. Metrikleri Kaydet
    mlflow.log_metric("last_volatility", data['volatility'].iloc[-1])
    print("Deney başarıyla kaydedildi!")